In [165]:
import pandas as pd
import os
import openmatrix as omx
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
interim_dir = "./data/interim/"
external_dir = "./data/external"

os.makedirs(interim_dir, exist_ok=True)

In [ ]:
# inputs
# target files
target_mode_share_file = os.path.join(external_dir, 'observed', 'mode_share_survey.csv')
target_trip_length_file = os.path.join(external_dir, 'observed', 'trip_length_survey.csv')
target_trip_length_spline_file = os.path.join(external_dir, 'observed', 'trip_length_spline_survey.csv')
target_trip_location_file = os.path.join(external_dir, 'observed', 'trip_location_survey.csv')
target_avg_trip_length_file = os.path.join(external_dir, 'observed', 'avg_trip_length_survey.csv')
target_msa_summary_file = os.path.join(external_dir, 'observed', 'msa_summary_survey.csv')

# model files
airport_trips_output_file = os.path.join(external_dir, 'final_santrips.csv')
skim_file = os.path.join(external_dir, 'traffic_skims_MD.omx')     
poi_file = os.path.join(external_dir, 'dest_poi.omx')
land_use_file = os.path.join(external_dir, 'land_use.san.csv')

In [168]:
# outputs
mode_share_comparison_output_file = os.path.join(interim_dir, 'mode-share-comparison.csv')
trip_length_comparison_output_file = os.path.join(interim_dir, 'trip-length-comparison.csv')
trip_length_spline_calibration_output_file = os.path.join(interim_dir, 'trip-length-spline-calibration-comparison.csv')
trip_location_comparison_output_file = os.path.join(interim_dir, 'trip-location-comparison.csv')
avg_trip_length_comparison_output_file = os.path.join(interim_dir, 'avg-trip-length-comparison.csv')
msa_summary_comparison_output_file = os.path.join(interim_dir, 'msa-summary-comparison.csv')

In [169]:
MAX_DISTANCE_TLFD = 60

mode_mapping = {
    'PARK_LOC1': 'Drive and Park',
    'PARK_LOC2': 'Drive and Park',
    'PARK_LOC3': 'Drive and Park',
    'PARK_LOC4': 'Drive and Park',
    'PARK_LOC5': 'Drive and Park',
    'CURB_LOC1': 'Pickup Dropoff',
    'CURB_LOC2': 'Pickup Dropoff',
    'CURB_LOC3': 'Pickup Dropoff',
    'CURB_LOC4': 'Pickup Dropoff',
    'CURB_LOC5': 'Pickup Dropoff',
    'SHUTTLEVAN': 'Shuttle Van',
    'RENTAL': 'Rental Car',
    'RIDEHAIL_LOC1': 'Ridehail',
    'TAXI_LOC1': 'Taxi',
    'WALK_LOC': 'Transit',
    'WALK_PRM': 'Transit',
    'WALK_MIX': 'Transit',
    'KNR_MIX': 'Transit',
    'KNR_PRM': 'Transit',
    'KNR_LOC': 'Transit'
}

purpose_mapping = {
    'vis_per': 'Visitor Personal',
    'vis_bus': 'Visitor Business',
    'res_per': 'Resident Personal',
    'res_bus': 'Resident Business',
    'external': 'External'
}

poi_location_mapping = {
    0: 'Other',
    1: 'La Jolla',
    2: 'Downtown', 
    3: 'Mission Valley'
}

msa_mapping = {
    1: 'Downtown',
    2: 'Central',
    3: 'North City',
    4: 'South Suburban',
    5: 'East Suburban',
    6: 'North County West',
    7: 'North County East',
    8: 'East County'
}

In [170]:
# read mode share and trip length frequency distribution for survey
mode_share_survey_df = pd.read_csv(target_mode_share_file)
trip_length_survey_df = pd.read_csv(target_trip_length_file)
trip_length_spline_survey_df = pd.read_csv(target_trip_length_spline_file).rename(columns={"distance_bin":"distance_bin_spline"})
trip_length_spline_survey_df["distance_bin_spline"] = trip_length_spline_survey_df["distance_bin_spline"].str.replace("-","_")
trip_location_survey_df = pd.read_csv(target_trip_location_file)
avg_trip_length_survey_df = pd.read_csv(target_avg_trip_length_file)
msa_summary_survey_df = pd.read_csv(target_msa_summary_file)

mode_share_survey_df['source'] = 'Survey'
trip_length_survey_df['source'] = 'Survey'
trip_length_spline_survey_df['source'] = 'Survey'
trip_location_survey_df['source'] = 'Survey'
avg_trip_length_survey_df['source'] = 'Survey'
msa_summary_survey_df['source'] = 'Survey'

In [171]:
# read airport output
airport_trips_df = pd.read_csv(airport_trips_output_file)

In [172]:
# read landuse data 
land_use_df = pd.read_csv(land_use_file)
taz_msa_xwalk = land_use_df[["taz","pseudomsa"]].drop_duplicates()

In [173]:
# read omx file and extract distance core 
skim_omx = omx.open_file(skim_file)
distance_skim = skim_omx['SOV_NT_L_DIST__MD']

# convert to long format
distance_df = pd.DataFrame(distance_skim).reset_index().melt(
    id_vars='index', 
    var_name='destination', 
    value_name='distance'
).rename(columns={'index': 'origin'})
distance_df['origin'] = distance_df['origin'] + 1
distance_df['destination'] = distance_df['destination'] + 1


In [174]:
# read poi file
poi_omx = omx.open_file(poi_file)
poi_dest = poi_omx['poi_dest']
poi_dest = np.array(poi_dest)

In [175]:
# Create a map of destinations to poi values for non-zero poi values
poi_dest_map = {}
for col_idx in range(poi_dest.shape[1]):  # Iterate through columns (destinations)
    poi_value = int(poi_dest[:, col_idx][0])
    if poi_value > 0:
        poi_dest_map[col_idx + 1] = poi_value  # +1 because omx has 0-based indexing


In [176]:
# filter trip_num == 1
# return trip with trip_num as 2 is added for drop-off modes
airport_trips_df = airport_trips_df[airport_trips_df['trip_num'] == 1]

# keep subset of columns 
columns_to_keep = ['person_id', 'household_id', 'tour_id', 'otaz', 'dtaz', 'party', 'outbound',
                   'weight_trip', 'weight_person_trip', 'primary_purpose', 'arrival_mode']

airport_trips_df = airport_trips_df[columns_to_keep]

In [177]:
# define simple mode
airport_trips_df['simple_mode'] = airport_trips_df['arrival_mode'].map(mode_mapping)

In [178]:
# Merge airport trips with distance based on origin and destination TAZs
airport_trips_df = airport_trips_df.merge(
    distance_df,
    left_on=['otaz', 'dtaz'],
    right_on=['origin', 'destination'],
    how='left'
)

In [179]:
# Cap distance at MAX_DISTANCE_TLFD
airport_trips_df['distance'] = airport_trips_df['distance'].clip(upper=MAX_DISTANCE_TLFD)

In [180]:
# get the non_airport_zone and identify the poi value (location dummy) for the non-airport zone 
airport_trips_df['non_airport_taz'] = np.where(
    airport_trips_df['outbound'] == 1, 
    airport_trips_df['otaz'],
    airport_trips_df['dtaz']
)

airport_trips_df['poi'] = airport_trips_df['non_airport_taz'].map(poi_dest_map).fillna(0).astype(int)

airport_trips_df['non_airport_location'] = airport_trips_df['poi'].map(poi_location_mapping).fillna('Other')

In [181]:
# join the msa information to the airport trips data
airport_trips_df = airport_trips_df.merge(
    taz_msa_xwalk,
    left_on='non_airport_taz',
    right_on='taz',
    how='left'
)

airport_trips_df['msa'] = airport_trips_df['pseudomsa'].map(msa_mapping).fillna('Other')

In [182]:
# Create distance bins of 5 miles from 0 to 50
distance_bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, np.inf]
distance_labels = ['0_5', '5_10', '10_15', '15_20', '20_25', '25_30', '30_35', '35_40', '40_45', '45_50', '50+']
airport_trips_df['distance_bin'] = pd.cut(
    airport_trips_df['distance'],
    bins=distance_bins,
    labels=distance_labels,
    right=False,
    include_lowest=True
)

In [183]:
# Create distance bins as per spline terms for calibration
distance_bins_spline = [0, 0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 7.5, 10.0, 15.0, 20.0, 30.0, 40.0, 50.0, np.inf]
distance_labels_spline = ['0_0.25', '0.25_0.5', '0.5_1', '1_2', '2_3', '3_4', '4_5', '5_7.5', '7.5_10', '10_15', '15_20', '20_30', '30_40', '40_50', '50+']
airport_trips_df['distance_bin_spline'] = pd.cut(
    airport_trips_df['distance'],
    bins=distance_bins_spline,
    labels=distance_labels_spline,
    right=True,
    include_lowest=True
)

In [184]:
airport_trips_df.head(10)

,person_id,household_id,tour_id,otaz,dtaz,party,outbound,weight_trip,weight_person_trip,primary_purpose,arrival_mode,simple_mode,origin,destination,distance,non_airport_taz,poi,non_airport_location,taz,pseudomsa,msa,distance_bin,distance_bin_spline
0,9545,17988,1,2946,1457,1,True,1,2,res_per,CURB_LOC1,Pickup Dropoff,2946,1457,13.058104,2946,0,Other,2946,3,North City,10_15,10_15
1,1609,30118,2,3036,1457,4,True,1,5,res_per,CURB_LOC1,Pickup Dropoff,3036,1457,7.926567,3036,0,Other,3036,2,Central,5_10,7.5_10
2,1054,20303,3,1472,1457,4,True,1,4,vis_bus,TAXI_LOC1,Taxi,1472,1457,0.854921,1472,2,Downtown,1472,2,Central,0_5,0.5_1
3,21133,4852,4,1818,1457,3,True,1,3,vis_bus,RIDEHAIL_LOC1,Ridehail,1818,1457,3.349836,1818,2,Downtown,1818,1,Downtown,0_5,3_4
4,17905,3400,5,3174,1457,1,True,1,2,res_per,CURB_LOC1,Pickup Dropoff,3174,1457,24.598885,3174,0,Other,3174,3,North City,20_25,20_30
5,28862,15547,6,3855,1457,1,True,1,2,res_per,CURB_LOC1,Pickup Dropoff,3855,1457,14.474214,3855,0,Other,3855,3,North City,10_15,10_15
6,29308,5987,7,2914,1457,1,True,1,1,res_per,RIDEHAIL_LOC1,Ridehail,2914,1457,18.767330,2914,0,Other,2914,3,North City,15_20,15_20
7,21548,17034,8,3268,1457,1,True,1,2,vis_bus,CURB_LOC1,Pickup Dropoff,3268,1457,14.509632,3268,0,Other,3268,4,South Suburban,10_15,10_15
8,30396,33328,9,2408,1457,1,True,1,1,vis_bus,RENTAL,Rental Car,2408,1457,17.249468,2408,0,Other,2408,4,South Suburban,15_20,15_20
9,23272,17669,10,2421,1457,3,True,1,3,vis_bus,RIDEHAIL_LOC1,Ridehail,2421,1457,9.122113,2421,3,Mission Valley,2421,3,North City,5_10,7.5_10


In [185]:
airport_trips_df.primary_purpose.value_counts()

primary_purpose
vis_per     12017
res_per      9325
vis_bus      6907
res_bus      3319
external     2426
Name: count, dtype: int64

In [186]:
test_df = airport_trips_df[airport_trips_df.primary_purpose == 'external']
test_df = test_df[['person_id', 'household_id', 'tour_id', 'otaz', 'dtaz', 'outbound', 'primary_purpose', 'arrival_mode', 'simple_mode', 'non_airport_taz', 'pseudomsa', 'msa']]
test_df.head(10)

,person_id,household_id,tour_id,otaz,dtaz,outbound,primary_purpose,arrival_mode,simple_mode,non_airport_taz,pseudomsa,msa
31568,28322,18355,31322,2227,1457,True,external,CURB_LOC1,Pickup Dropoff,2227,7,North County East
31569,28652,6499,31323,14,1457,True,external,SHUTTLEVAN,Shuttle Van,14,6,North County West
31570,17314,13322,31324,3386,1457,True,external,CURB_LOC1,Pickup Dropoff,3386,7,North County East
31571,4627,8776,31325,3742,1457,True,external,CURB_LOC1,Pickup Dropoff,3742,4,South Suburban
31572,30719,3773,31326,2227,1457,True,external,RIDEHAIL_LOC1,Ridehail,2227,7,North County East
31573,24974,23273,31327,4943,1457,True,external,CURB_LOC1,Pickup Dropoff,4943,8,East County
31574,22068,1349,31328,3742,1457,True,external,CURB_LOC1,Pickup Dropoff,3742,4,South Suburban
31575,13606,7756,31329,14,1457,True,external,PARK_LOC1,Drive and Park,14,6,North County West
31576,25630,5494,31330,2227,1457,True,external,SHUTTLEVAN,Shuttle Van,2227,7,North County East
31577,33049,1513,31331,2227,1457,True,external,CURB_LOC1,Pickup Dropoff,2227,7,North County East


In [187]:
# simulated mode share
mode_share_simulated_df = (airport_trips_df
    .groupby(['primary_purpose', 'arrival_mode', 'simple_mode'])
    .agg(trips=('weight_trip', 'sum'))
    .reset_index()
)

mode_share_simulated_df['total_share'] = round(mode_share_simulated_df['trips'] / mode_share_simulated_df['trips'].sum(), 4)

mode_share_simulated_df['purpose_share'] = (mode_share_simulated_df
    .groupby('primary_purpose')['trips']
    .transform(lambda x: round(x / x.sum(), 4))
)

mode_share_simulated_df['source'] = 'ABM3'

mode_share_simulated_df = mode_share_simulated_df[['primary_purpose', 'arrival_mode', 'simple_mode', 'total_share', 'purpose_share', 'source']]

In [188]:
# simulated trip length frequency distribution
trip_length_simulated_df = (airport_trips_df
    .groupby(['primary_purpose', 'distance_bin'], observed=True)
    .agg(trips=('weight_trip', 'sum'))
    .reset_index()
)

trip_length_simulated_df['total_share'] = round(trip_length_simulated_df['trips'] / trip_length_simulated_df['trips'].sum(), 4)

trip_length_simulated_df['purpose_share'] = (trip_length_simulated_df
    .groupby('primary_purpose')['trips']
    .transform(lambda x: round(x / x.sum(), 4))
)

trip_length_simulated_df['source'] = 'ABM3'

trip_length_simulated_df = trip_length_simulated_df[['primary_purpose', 'distance_bin', 'total_share', 'purpose_share', 'source']]

In [189]:
# simulated trip length frequency distribution
trip_length_spline_simulated_df = (airport_trips_df
    .groupby(['primary_purpose', 'distance_bin_spline'], observed=True)
    .agg(trips=('weight_trip', 'sum'))
    .reset_index()
)

trip_length_spline_simulated_df['total_share'] = round(trip_length_spline_simulated_df['trips'] / trip_length_spline_simulated_df['trips'].sum(), 4)

trip_length_spline_simulated_df['purpose_share'] = (trip_length_spline_simulated_df
    .groupby('primary_purpose')['trips']
    .transform(lambda x: round(x / x.sum(), 4))
)

trip_length_spline_simulated_df['source'] = 'ABM3'

trip_length_spline_simulated_df = trip_length_spline_simulated_df[['primary_purpose', 'distance_bin_spline', 'total_share', 'purpose_share', 'source']]

In [190]:
# simulated non-airport location 
trip_location_simulated_df = (airport_trips_df
    .groupby(['primary_purpose', 'non_airport_location'], observed=True)
    .agg(trips=('weight_trip', 'sum'))
    .reset_index()
)

trip_location_simulated_df['total_share'] = round(trip_location_simulated_df['trips'] / trip_location_simulated_df['trips'].sum(), 4)

trip_location_simulated_df['purpose_share'] = (trip_location_simulated_df
    .groupby('primary_purpose')['trips']
    .transform(lambda x: round(x / x.sum(), 4))
)

trip_location_simulated_df['source'] = 'ABM3'

trip_location_simulated_df = trip_location_simulated_df[['primary_purpose', 'non_airport_location', 'total_share', 'purpose_share', 'source']]

In [191]:
# simulated average trip length
avg_trip_length_simulated_df = (airport_trips_df
    .groupby(['primary_purpose'], observed=True)
    .agg(avg_trip_length=('distance', lambda x: round(np.average(x), 2)))
    .reset_index()
)

avg_trip_length_simulated_df['source'] = 'ABM3'

In [192]:
# trip destination by MSA
# drop externals and consider only outbound trips. 
msa_summary_simulated_df = airport_trips_df[airport_trips_df['primary_purpose'] != 'external']
msa_summary_simulated_df = msa_summary_simulated_df[msa_summary_simulated_df['outbound']]
msa_summary_simulated_df['purpose'] = np.where(
    msa_summary_simulated_df['primary_purpose'].isin(['res_bus', 'res_per']),
    "Resident",
    "Visitor"
)

msa_summary_simulated_df = (msa_summary_simulated_df
    .groupby(['purpose', 'msa'], observed=True)
    .agg(trips=('weight_trip', 'sum'))
    .reset_index()
)

msa_summary_simulated_df['total_share'] = round(msa_summary_simulated_df['trips'] / msa_summary_simulated_df['trips'].sum(), 4)

msa_summary_simulated_df['purpose_share'] = (msa_summary_simulated_df
    .groupby('purpose')['trips']
    .transform(lambda x: round(x / x.sum(), 4))
)

msa_summary_simulated_df['source'] = 'ABM3'

msa_summary_simulated_df = msa_summary_simulated_df[['purpose', 'msa', 'total_share', 'purpose_share', 'source']]

In [193]:
# Combine simulated and survey mode share data
mode_share_df = pd.concat([mode_share_simulated_df, mode_share_survey_df], ignore_index=True)
mode_share_df = mode_share_df.fillna(0)

# Combine simulated and survey trip length data
trip_length_df = pd.concat([trip_length_simulated_df, trip_length_survey_df], ignore_index=True)
trip_length_df = trip_length_df.fillna(0)

# Combine simulated and survey trip length spline data
trip_length_spline_df = pd.concat([trip_length_spline_simulated_df, trip_length_spline_survey_df], ignore_index=True)
trip_length_spline_df = trip_length_spline_df.fillna(0)

# Combine simulated and survey trip location data
trip_location_df = pd.concat([trip_location_simulated_df, trip_location_survey_df], ignore_index=True)
trip_location_df = trip_location_df.fillna(0)

# Combine simulated and survey average trip length data
avg_trip_length_df = pd.concat([avg_trip_length_simulated_df, avg_trip_length_survey_df], ignore_index=True)
avg_trip_length_df = avg_trip_length_df.fillna(0)

# Combine simulated and survey msa summary
msa_summary_df = pd.concat([msa_summary_simulated_df, msa_summary_survey_df], ignore_index=True)
msa_summary_df = msa_summary_df.fillna(0)

In [194]:
mode_share_df.head()

,primary_purpose,arrival_mode,simple_mode,total_share,purpose_share,source
0,external,CURB_LOC1,Pickup Dropoff,0.0305,0.4267,ABM3
1,external,KNR_LOC,Transit,0.0001,0.0008,ABM3
2,external,KNR_MIX,Transit,0.0002,0.0025,ABM3
3,external,KNR_PRM,Transit,0.0001,0.0012,ABM3
4,external,PARK_LOC1,Drive and Park,0.0053,0.0739,ABM3


In [195]:
trip_length_df.head()

,primary_purpose,distance_bin,total_share,purpose_share,source
0,external,15_20,0.0258,0.3611,ABM3
1,external,20_25,0.0028,0.0392,ABM3
2,external,40_45,0.0034,0.0474,ABM3
3,external,50+,0.0394,0.5523,ABM3
4,res_bus,0_5,0.0089,0.0916,ABM3


In [196]:
trip_length_spline_df.head()

,primary_purpose,distance_bin_spline,total_share,purpose_share,source
0,external,15_20,0.0258,0.3611,ABM3
1,external,20_30,0.0028,0.0392,ABM3
2,external,40_50,0.0034,0.0474,ABM3
3,external,50+,0.0394,0.5523,ABM3
4,res_bus,0.25_0.5,0.0000,0.0003,ABM3


In [197]:
trip_location_df.head()

,primary_purpose,non_airport_location,total_share,purpose_share,source
0,external,Other,0.0714,1.0000,ABM3
1,res_bus,Downtown,0.0049,0.0506,ABM3
2,res_bus,La Jolla,0.0049,0.0506,ABM3
3,res_bus,Mission Valley,0.0043,0.0440,ABM3
4,res_bus,Other,0.0835,0.8548,ABM3


In [198]:
avg_trip_length_df

,primary_purpose,avg_trip_length,source
0,external,41.52,ABM3
1,res_bus,17.74,ABM3
2,res_per,16.67,ABM3
3,vis_bus,7.70,ABM3
4,vis_per,12.40,ABM3
5,external,44.20,Survey
6,res_bus,18.28,Survey
7,res_per,16.97,Survey
8,vis_bus,8.47,Survey
9,vis_per,12.92,Survey


In [199]:
msa_summary_df.head()

,purpose,msa,total_share,purpose_share,source
0,Resident,Central,0.0845,0.2097,ABM3
1,Resident,Downtown,0.0193,0.0479,ABM3
2,Resident,East County,0.0013,0.0032,ABM3
3,Resident,East Suburban,0.0487,0.1209,ABM3
4,Resident,North City,0.1172,0.2911,ABM3


In [200]:
# Recode primary purpose
mode_share_df['primary_purpose'] = mode_share_df['primary_purpose'].map(purpose_mapping)
trip_length_df['primary_purpose'] = trip_length_df['primary_purpose'].map(purpose_mapping)
trip_length_spline_df['primary_purpose'] = trip_length_spline_df['primary_purpose'].map(purpose_mapping)
trip_location_df['primary_purpose'] = trip_location_df['primary_purpose'].map(purpose_mapping)
avg_trip_length_df['primary_purpose'] = avg_trip_length_df['primary_purpose'].map(purpose_mapping)

# Convert categorical distance_bin to string
trip_length_df['distance_bin'] = trip_length_df['distance_bin'].astype(str)
trip_length_spline_df['distance_bin_spline'] = trip_length_spline_df['distance_bin_spline'].astype(str)


In [201]:
mode_share_df.to_csv(mode_share_comparison_output_file, index=False)

trip_length_df.to_csv(trip_length_comparison_output_file, index=False)

trip_length_spline_df.to_csv(trip_length_spline_calibration_output_file, index=False)

trip_location_df.to_csv(trip_location_comparison_output_file, index=False)

avg_trip_length_df.to_csv(avg_trip_length_comparison_output_file, index=False)

msa_summary_df.to_csv(msa_summary_comparison_output_file, index=False)